[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/06_ai_engineering/A1_llm_providers_guide.ipynb)

> 📎 **Appendix notebook — reference style.** This is one of the optional appendices (see `README.md`). Unlike the main course notebooks, appendices are written as a demo / reference: they focus on *seeing* a library at work rather than on interactive exercises. You won't find the full Solution / Debug-me / Self-assessment scaffolding here. Each appendix is built to run end-to-end *without* the optional library — it falls back to a small built-in stand-in (or skips the library-specific cells), so you can read and run it offline. Install the optional library (see the **Install** section below) to swap the stand-in for the real thing.

---
# 📓 Notebook A1 — LLM Providers Guide

> **Module:** AI Engineering · **Type:** Appendix · **Estimated time:** 30–45 min · **Difficulty:** Intermediate

By default every AI notebook in this course uses an offline `MockLLM` so you can run the code without internet, an API key, or a credit card. That's perfect for *learning the patterns*. The moment you want **real intelligence** in the answers, you swap one line — and this notebook is the reference for that swap.

You'll learn how to use:

- 🟢 **OpenAI** — the most popular hosted API; great default for most work.
- 🟠 **Anthropic** (Claude) — strong on reasoning, long context, careful tone.
- 🔵 **Google** (Gemini) — competitive on cost, especially for high-volume tasks.
- 🟣 **Ollama** — run *local* open-source LLMs on your own machine. No internet, no per-call cost.

All four implementations live in [`llm_providers.py`](../llm_providers.py) at the root of the course repo and share the **same `chat()` interface**. Swapping providers in any of the AI notebooks (NB 21–26, and the NB 42 capstone) is a *one-line change*.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Pick the right provider for cost, latency, quality, and data-privacy constraints.
2. Install and authenticate each of the four providers safely.
3. Use the unified `llm_providers` module to swap providers without changing application code.
4. Estimate **cost** for a known workload across providers.
5. Run a fully **local** open-source LLM via Ollama.
6. Use the same pattern for **embeddings** (NB 23) — hosted vs local.

## ✅ Prerequisites

NB 22 (AI workflows). Helpful: NB 23 (embeddings), NB 26 (evaluation).

## 1. The unified `chat()` interface

Every provider in this course implements one minimal method:

```python
response = llm.chat(
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user",   "content": "Hello!"},
    ],
    temperature=0.0,
    max_tokens=512,
)

response["text"]        # the assistant's reply (str)
response["model"]       # actual model name returned
response["tokens_in"]   # input tokens billed (or estimated)
response["tokens_out"]  # output tokens billed
response["latency_s"]   # wall-clock time, seconds
```

That contract is the same for all five classes — `MockLLM`, `OpenAILLM`, `AnthropicLLM`, `GoogleLLM`, `OllamaLLM`. Your notebook code does not change when you swap them; only the *constructor call* changes.

### 🔬 What actually happens — one interface, many swappable backends

The intro above says *"your code doesn't change when you swap providers."* That sounds like magic. It isn't — it's a single, very old object-oriented idea called **program to the interface, not the implementation**. Let's see the gears turn.

The problem: every vendor ships a *different* SDK. OpenAI wants `client.chat.completions.create(...)`, Anthropic wants `client.messages.create(...)`, Google wants `model.generate_content(...)`. If your app calls those directly, your app is **welded to one vendor**. Swapping = rewriting.

The fix: put ONE thin layer in the middle that every vendor must satisfy.

```text
                    ┌─────────────────────────────────────┐
   your app code    │   summarize(client, text)           │   ← calls ONLY .chat()
   (never changes)  │   ...    client.chat(prompt)   ...  │     never names a vendor
                    └──────────────────┬──────────────────┘
                                       │  "give me a .chat(prompt)"
                                       ▼
                    ┌─────────────────────────────────────┐
   the INTERFACE    │   class LLMClient:                  │   ← the contract / blueprint
   (the contract)   │       def chat(self, prompt): ...   │     .chat() raises until a
                    └──────┬───────────────────────┬──────┘     subclass fills it in
                           │  subclass             │  subclass
                ┌──────────▼─────────┐   ┌──────────▼─────────┐
   the BACKENDS │  EchoProvider      │   │  UpperProvider     │   ← each implements
   (swappable)  │  def chat(): ...   │   │  def chat(): ...   │     .chat() its own way
                └────────────────────┘   └────────────────────┘
                   (≈ OpenAILLM)            (≈ AnthropicLLM)
```

Your app talks to the **box in the middle**. The boxes at the bottom are interchangeable, because they all expose the same `.chat()` shape. Add a new vendor? Add one new box at the bottom. The app — and that's the whole point — *never finds out*.


**Step by step — how Python makes the swap invisible.**

**Step 1 — the base class declares the contract, but refuses to do the work:**

```python
class LLMClient:
    def chat(self, prompt):
        raise NotImplementedError   # "every provider MUST supply its own"
```

`raise NotImplementedError` is a promise written in code: *if you subclass me and forget `chat`, you'll get a loud, clear error* — not a silent wrong answer.

**Step 2 — each provider is a subclass that fills in `chat`:**

```python
class EchoProvider(LLMClient):
    def chat(self, prompt): ...     # talks to "vendor A"

class UpperProvider(LLMClient):
    def chat(self, prompt): ...     # talks to "vendor B"
```

`class EchoProvider(LLMClient):` — the parent in parentheses means *"I am an `LLMClient`, plus my own `chat`."*

**Step 3 — the app is written against the *base type*, not any vendor:**

```python
def summarize(client, text):        # `client` is "some LLMClient", we don't care which
    return client.chat(f"Summarize: {text}")
```

When Python runs `client.chat(...)`, it looks at the **actual object** you passed and runs *that* class's `chat`. Pass an `EchoProvider` → Echo's `chat` runs. Pass an `UpperProvider` → Upper's `chat` runs. Same line of app code, different backend. That late, run-time "which `chat`?" decision is called **dynamic dispatch**, and it's the engine under the whole pattern.


In [1]:
# 🧪 OFFLINE PROOF — no SDK, no API key, stdlib only.
# Base interface + two mock provider subclasses + one app function that
# works UNCHANGED with either. This is exactly the shape of llm_providers.py.

class LLMClient:
    """The interface. Every provider must satisfy this shape."""
    def __init__(self, model):
        self.model = model

    def chat(self, prompt):
        # The base class deliberately does NOT know how to chat.
        raise NotImplementedError("every provider subclass must implement chat()")

    def __repr__(self):
        # type(self).__name__ prints the SUBCLASS name automatically.
        return f"{type(self).__name__}(model={self.model!r})"

In [2]:
class EchoProvider(LLMClient):
    """Mock 'vendor A' — echoes the prompt back."""
    def chat(self, prompt):
        return f"[echo:{self.model}] you said: {prompt}"


class UpperProvider(LLMClient):
    """Mock 'vendor B' — SHOUTS the prompt back."""
    def chat(self, prompt):
        return f"[upper:{self.model}] {prompt.upper()}"

In [3]:
# --- The application code. Notice: it never mentions Echo or Upper. ---
def summarize(client, text):
    """Pretend-summarize `text` using ANY object that is an LLMClient."""
    return client.chat(f"Summarize in one line: {text}")

In [4]:
# Swap the backend by changing ONLY the constructor — app call is identical.
text = "Quarterly revenue rose while churn fell for the third month running."

for client in (EchoProvider(model="echo-1"), UpperProvider(model="upper-1")):
    print(client)                        # __repr__ shows the real subclass
    print("  ->", summarize(client, text))

# Proof that the base class refuses to answer on its own:
try:
    LLMClient(model="base").chat("hi")
except NotImplementedError as e:
    print("\nbase LLMClient.chat() correctly refused:", e)

EchoProvider(model='echo-1')
  -> [echo:echo-1] you said: Summarize in one line: Quarterly revenue rose while churn fell for the third month running.
UpperProvider(model='upper-1')
  -> [upper:upper-1] SUMMARIZE IN ONE LINE: QUARTERLY REVENUE ROSE WHILE CHURN FELL FOR THE THIRD MONTH RUNNING.

base LLMClient.chat() correctly refused: every provider subclass must implement chat()


**Read the output.** One `summarize(client, text)` call produced two different results — because `client.chat(...)` dispatched to whichever subclass you handed it. The app function has **zero knowledge** of `EchoProvider` or `UpperProvider`. And the bare `LLMClient` raised `NotImplementedError`, proving the base is a *contract*, not a usable provider.

#### Interface (the contract) vs. Implementation (the backend)

| | Interface — `LLMClient` | Implementation — `EchoProvider` / `UpperProvider` |
|---|---|---|
| Role | Defines the *shape* every provider must have | Provides the *actual behaviour* |
| `chat()` body | `raise NotImplementedError` | Real logic (calls vendor A / vendor B) |
| What the app depends on | ✅ this — `client.chat(prompt)` | ❌ never named in app code |
| How many | One, shared | One per vendor; add freely |
| To add a new provider | unchanged | write **one** new subclass |

🎯 **The payoff, quantified.** Adding Google Gemini to a real app = **one new subclass** (`class GoogleLLM(LLMClient)`) and **one changed line** (the constructor you call). App functions like `summarize`, your prompts, your post-processing: **0 changes**. That ratio — one new file vs. zero touched files — is *why* a thin abstraction layer is worth its weight.


🧠 **Mental model — code against the interface, not the vendor.**

Think of `LLMClient` as a **power socket** on the wall. Your app is the appliance with a plug. The socket guarantees a fixed shape (`.chat(prompt) -> response`); the power station behind it — coal, solar, nuclear — is none of the appliance's business. Swapping the power source never means re-wiring the toaster.

> ⚠️ **When it's overkill.** If you will *only ever* call one vendor and never test offline, a direct SDK call is fine — don't build an abstraction for an axis that never moves. The layer earns its keep the moment you need a **second** backend: a `MockLLM` for fast offline tests, a cheap local model in dev and a premium one in prod, or an A/B comparison of two vendors (exactly Section 8 of this notebook).

This is precisely how the course's `llm_providers.py` is wired: `MockLLM`, `OpenAILLM`, `AnthropicLLM`, `GoogleLLM`, and `OllamaLLM` are all subclasses sharing the one `chat()` contract you saw in Section 1 — so every other notebook in the course just calls `.chat()` and works with whichever provider you plugged in.


## 2. Smoke test — the `MockLLM` works offline

Let's first verify the module imports and the offline mock works. Everything else in this notebook is *reference code* that you only run if you have the corresponding key/SDK.

In [5]:
# Make the repo-root llm_providers.py importable regardless of which folder
# Jupyter launched the kernel in (it lives at the course root).
import sys, pathlib
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "llm_providers.py").exists()), None)
if _root:                       # local checkout — use the repo's module
    sys.path.insert(0, str(_root))
else:                           # Colab / standalone — fetch it next to the notebook
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/ChrisW09/Python-for-AI-Driven-Automation"
        "/main/llm_providers.py", "llm_providers.py")

from llm_providers import MockLLM, get_llm

llm = MockLLM()
r = llm.chat(messages=[
    {"role": "system", "content": "Return JSON with keys sentiment and topic."},
    {"role": "user",   "content": "My invoice is wrong, please refund."},
])
print(f"text       : {r['text']}")
print(f"model      : {r['model']}")
print(f"tokens_in  : {r['tokens_in']}")
print(f"tokens_out : {r['tokens_out']}")
print(f"latency_s  : {r['latency_s']:.4f}")


text       : {"sentiment": "negative", "topic": "billing"}
model      : mock-mini
tokens_in  : 16
tokens_out : 17
latency_s  : 0.0002


## 3. Picking a provider — the decision table

| Criterion | OpenAI | Anthropic | Google | Ollama (local) |
|---|---|---|---|---|
| **Latency** (small models) | ~0.5–2 s | ~0.5–2 s | ~0.5–1.5 s | depends on hardware |
| **Per-token cost** | low–medium | low–medium | very low (Flash) | $0 |
| **Quality (small models)** | gpt-5.4-mini ★★★★ | claude-haiku ★★★★ | gemini-flash ★★★ | Llama 3.2 3B ★★ |
| **Quality (flagship)** | gpt-5.5 ★★★★★ | claude-sonnet ★★★★★ | gemini-pro ★★★★ | Llama 3.1 70B ★★★★ |
| **Long-context** | 400K | 1M | 1M+ | depends on model |
| **Data privacy** | sent to API | sent to API | sent to API | **stays local** |
| **Internet required** | yes | yes | yes | **no** |
| **API key required** | yes | yes | yes | **no** |
| **Best for** | default; reliable, well-documented | reasoning-heavy; longer drafts; careful tone | high-volume / cost-sensitive | privacy-sensitive; offline; tinkering |

### When to pick which

- **Default? OpenAI.** Largest ecosystem, best documentation, mature SDK.
- **Need to process *long* documents?** Anthropic (1M context) or Gemini (1M).
- **Cost-sensitive at high volume?** Gemini Flash is often cheapest.
- **Data must not leave your machine?** Ollama. Period.
- **Demoing without any keys?** Ollama (or stay on `MockLLM`).


---

### ✋ Quick exercise (~2 min) — Read the decision table

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A hospital wants to classify patient messages, but regulation says the data **must never leave the building** (no internet, no third-party API). Using only the §3 decision table, write a tiny `pick_provider(needs_local)` that returns the right provider name for this case — and the sensible default when local-only isn't required.

```python
def pick_provider(needs_local: bool) -> str:
    ...
```

In [6]:
# ✍️ Your turn 👇
# A hospital wants to classify patient messages, but the data must NEVER
# leave its building. Use the §3 decision table to pick a provider.
def pick_provider(needs_local: bool) -> str:
    ...

# print(pick_provider(needs_local=True))


<details>
<summary>✅ <b>Solution</b></summary>

```python
def pick_provider(needs_local: bool) -> str:
    # From the §3 table: only Ollama keeps data on the machine
    # (no internet, no API key). Everything else is sent to an API.
    return "Ollama" if needs_local else "OpenAI"

print(pick_provider(needs_local=True))   # -> Ollama  (hospital case)
print(pick_provider(needs_local=False))  # -> OpenAI  (safe default)
```

The hospital's data "must not leave the machine", so the decision table points to **Ollama** — the only row with *Data privacy = stays local* and *Internet required = no*. When privacy isn't the binding constraint, the table's recommended default is **OpenAI**.
</details>

## 4. 🟢 OpenAI — the default

```bash
pip install openai
export OPENAI_API_KEY=sk-...        # in your shell, BEFORE launching Jupyter
```

> ⚠️ **Never** paste an API key into a notebook cell that you might commit to git. Use environment variables (or a `.env` file with `python-dotenv`).

In [7]:
# Reference code — uncomment after `pip install openai` and setting the key.
# from llm_providers import OpenAILLM
#
# llm = OpenAILLM(model="gpt-5.4-mini")
# r = llm.chat(messages=[
#     {"role": "system", "content": "You are a concise assistant."},
#     {"role": "user",   "content": "Why are LLMs good at translation?"},
# ])
# print(r["text"])
# print(f"\nTokens: in={r['tokens_in']}, out={r['tokens_out']}  |  cost ≈ ${r['tokens_in']*0.00075/1000 + r['tokens_out']*0.0045/1000:.5f}")

print("(cell intentionally inactive — uncomment after install + auth)")


(cell intentionally inactive — uncomment after install + auth)


### Common OpenAI models

| Model | Speed | Cost | Notes |
|---|---|---|---|
| `gpt-5.4-mini` | very fast | cheap | Default choice. Excellent quality-per-dollar. |
| `gpt-5.5` | fast | medium–high | Current flagship; smartest of the chat models; long context. |
| `gpt-5.4-nano` | very fast | cheapest | Short, simple, high-volume tasks. |
| `text-embedding-3-small` | very fast | very cheap | The embeddings model from NB 23. |

> 💡 **Picking the smallest model that works** is the single biggest cost lever. `gpt-5.4-mini` handles 80% of classification / summarisation tasks fine.

## 5. 🟠 Anthropic (Claude)

```bash
pip install anthropic
export ANTHROPIC_API_KEY=sk-ant-...
```

Anthropic's API splits the system prompt into its own `system=` parameter. The wrapper in `llm_providers.py` does that translation for you — your code stays in the OpenAI-style format.

In [8]:
# Reference code — same shape as OpenAILLM.
# from llm_providers import AnthropicLLM
#
# llm = AnthropicLLM(model="claude-haiku-4-5")
# r = llm.chat(messages=[
#     {"role": "system", "content": "You are a careful copy editor."},
#     {"role": "user",   "content": "Improve this: 'we will done that quickly'."},
# ])
# print(r["text"])

print("(cell intentionally inactive — uncomment after install + auth)")


(cell intentionally inactive — uncomment after install + auth)


### Common Anthropic models

| Model | Speed | Cost | Notes |
|---|---|---|---|
| `claude-haiku-4-5` | very fast | cheap | Great default for classification + light reasoning. |
| `claude-sonnet-5` | fast | medium | Current-generation Sonnet; best balance for production. |
| `claude-opus-4-8` | slower | high | Reasoning-heavy tasks, longer drafts, agentic work. |

Claude's strengths: **long context** (up to 1M tokens on Opus/Sonnet), careful tone in long-form writing, well-behaved JSON output.

## 6. 🔵 Google (Gemini)

```bash
pip install google-generativeai
export GOOGLE_API_KEY=...     # or GEMINI_API_KEY — both supported by the wrapper
```

The wrapper translates OpenAI-style messages into Gemini's `parts` format, and surfaces system instructions via the `system_instruction` parameter.

In [9]:
# Reference code — same call shape as OpenAILLM/AnthropicLLM.
# from llm_providers import GoogleLLM
#
# llm = GoogleLLM(model="gemini-2.5-flash")
# r = llm.chat(messages=[
#     {"role": "system", "content": "Return one JSON object with keys sentiment and topic."},
#     {"role": "user",   "content": "Add support for Markdown please."},
# ])
# print(r["text"])

print("(cell intentionally inactive — uncomment after install + auth)")


(cell intentionally inactive — uncomment after install + auth)


### Common Gemini models

| Model | Speed | Cost | Notes |
|---|---|---|---|
| `gemini-2.5-flash` | very fast | very cheap | Default; great for high-volume classification. |
| `gemini-2.5-flash-lite` | very fast | cheapest | Lighter — fine for short prompts. |
| `gemini-2.5-pro` | slower | medium | Smartest stable Gemini; **1M+ token context**. |

Gemini's superpower: that **1M-token context window**. Whole codebases or hour-long meeting transcripts fit in one prompt.

> ⚠️ Model generations move fast: `gemini-2.0-flash` (this notebook's previous default) was shut down in June 2026, and the Gemini 3.x generation (e.g. `gemini-3-flash-preview`) is rolling out. Check [the pricing page](https://ai.google.dev/gemini-api/docs/pricing) for the current lineup.

## 7. 🟣 Ollama — fully local LLMs

Ollama gives you the *same chat-style API* against open-source models running entirely on your own machine. **No internet. No API key. No per-call cost.** It is the right answer when:

- The data is sensitive and must not leave the box.
- You want to demo without any signup.
- You're experimenting with prompts and want zero variable cost.

### One-time setup

```bash
# 1. Install the Ollama server
#    https://ollama.com — installer for macOS / Linux / Windows.

# 2. Pull a model (the first call is slow; subsequent are local)
ollama pull llama3.2:3b              # 2 GB, runs on CPU
ollama pull qwen2.5:7b               # 4 GB, better quality
ollama pull deepseek-r1:8b           # reasoning-heavy

# 3. (Ollama serves on http://localhost:11434 by default — no action needed.)

# 4. Install the Python client
pip install ollama
```

In [10]:
# Reference code — uncomment after Ollama is running and a model is pulled.
# from llm_providers import OllamaLLM
#
# llm = OllamaLLM(model="llama3.2:3b")
# r = llm.chat(messages=[
#     {"role": "system", "content": "Reply in JSON with key 'sentiment' only."},
#     {"role": "user",   "content": "Loving the new dashboard."},
# ])
# print(r["text"])
# print(f"\nLatency: {r['latency_s']:.2f}s  (CPU-only is ~1-5s; GPU brings it under 1s)")

print("(cell intentionally inactive — uncomment after Ollama is running)")


(cell intentionally inactive — uncomment after Ollama is running)


### Which local model should I pick?

| Model | Disk | RAM/VRAM | Quality (rough) | Good for |
|---|---|---|---|---|
| `llama3.2:3b` | 2 GB | 4 GB | ★★ | Demos, classification, short replies |
| `qwen2.5:7b` | 4 GB | 8 GB | ★★★ | Most light-AI work; good multilingual |
| `llama3.1:8b` | 5 GB | 8 GB | ★★★ | Stronger reasoning |
| `gemma2:9b` | 5 GB | 10 GB | ★★★ | Strong instruction-following |
| `qwen2.5:32b` | 20 GB | 32 GB | ★★★★ | Production-quality (needs a GPU) |

> 💡 **Start with the 3B model.** It runs on a laptop CPU and is good enough for prompt-engineering experiments. Move up only if quality is the bottleneck.

---

### ✋ Quick exercise (~2 min) — Size a local model to the hardware

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A colleague wants to run a local Ollama model on a laptop with **8 GB of RAM**. Using the RAM figures from the §7 table (provided below), filter the models that fit and pick the strongest one your hardware can load.

```python
LOCAL_MODELS = {  # model -> RAM/VRAM needed (GB), from the §7 table
    "llama3.2:3b": 4, "qwen2.5:7b": 8, "llama3.1:8b": 8,
    "gemma2:9b": 10, "qwen2.5:32b": 32,
}
```

In [11]:
# ✍️ Your turn 👇
# RAM/VRAM needs from the §7 "which local model" table:
LOCAL_MODELS = {
    "llama3.2:3b": 4,
    "qwen2.5:7b":  8,
    "llama3.1:8b": 8,
    "gemma2:9b":   10,
    "qwen2.5:32b": 32,
}
# A laptop has 8 GB. Which models fit? Of those, pick the strongest one.


<details>
<summary>✅ <b>Solution</b></summary>

```python
RAM_BUDGET_GB = 8
fits = [m for m, ram in LOCAL_MODELS.items() if ram <= RAM_BUDGET_GB]
print("Fits in 8 GB:", fits)
# Highest-RAM model that still fits ≈ the strongest one you can run
best = max(fits, key=LOCAL_MODELS.get)
print("Pick:", best)   # -> qwen2.5:7b
```

Three models fit in 8 GB (`llama3.2:3b`, `qwen2.5:7b`, `llama3.1:8b`); `gemma2:9b` (10 GB) and `qwen2.5:32b` (32 GB) don't. Two of them tie at exactly 8 GB — the most your laptop can load — and `max()` returns the **first** one it meets, `qwen2.5:7b` (`llama3.1:8b` is an equally valid pick at the same RAM). Either is the §7 guidance for "move up only when quality is the bottleneck".
</details>

## 8. Comparing two providers on the same task

Once your application uses the unified interface, **A/B-testing models becomes trivial** — same loop, two different clients. The cell below shows the pattern (using `MockLLM` twice; in real life you'd use two different providers).

In [12]:
# A/B template: same prompts, two providers, compare outputs.
from llm_providers import MockLLM

provider_a = MockLLM(seed=0)
provider_b = MockLLM(seed=42)              # in real life: OpenAILLM() and AnthropicLLM()

queries = [
    "Refund please.",
    "Why is loading so slow?",
    "Could you add CSV export?",
]

In [13]:
# Run both providers over every query, collecting their answers.
rows = []
for q in queries:
    a = provider_a.chat(messages=[
        {"role": "system", "content": "Return JSON with sentiment and topic."},
        {"role": "user",   "content": q},
    ])
    b = provider_b.chat(messages=[
        {"role": "system", "content": "Return JSON with sentiment and topic."},
        {"role": "user",   "content": q},
    ])
    rows.append((q, a["text"], b["text"]))

In [14]:
# Compare the two providers side by side.
for q, ra, rb in rows:
    print(f"Q: {q}")
    print(f"  A → {ra}")
    print(f"  B → {rb}")
    print()

Q: Refund please.
  A → {"sentiment": "negative", "topic": "billing"}
  B → {"sentiment": "negative", "topic": "billing"}

Q: Why is loading so slow?
  A → {"sentiment": "negative", "topic": "tech"}
  B → {"sentiment": "negative", "topic": "tech"}

Q: Could you add CSV export?
  A → {"sentiment": "neutral", "topic": "feature"}
  B → {"sentiment": "neutral", "topic": "feature"}



**Why this matters.** The same `messages` list works against any provider; the same output dict comes back. You can keep your prompts and evaluation harness *exactly* as in NB 26 and swap models for a head-to-head comparison.

> 🎯 **Evaluation-driven model selection.** Run your golden set (NB 26) against each provider, compare accuracy / latency / cost. The "best" model is the cheapest one that passes the eval — not necessarily the most expensive one.

## 9. Cost estimation — back-of-envelope per provider

A simple model: cost = `(tokens_in / 1000) * price_in + (tokens_out / 1000) * price_out`. Approximate prices (USD per 1K tokens, check the provider for current values — they change):

In [15]:
# Approximate per-1K-token prices in USD (rough snapshot figures — prices change; verify with the provider!)
PRICES = {
    "OpenAI gpt-5.4-mini":         {"in": 0.000750, "out": 0.004500},
    "OpenAI gpt-5.5":              {"in": 0.005000, "out": 0.030000},
    "Anthropic claude-haiku":      {"in": 0.001000, "out": 0.005000},
    "Anthropic claude-sonnet":     {"in": 0.003000, "out": 0.015000},
    "Google gemini-2.5-flash":     {"in": 0.000300, "out": 0.002500},
    "Google gemini-2.5-pro":       {"in": 0.001250, "out": 0.010000},  # ≤200K-token prompts; more above
    "Ollama llama3.2:3b (local)":  {"in": 0.000000, "out": 0.000000},
}

# A realistic monthly workload: 50,000 messages × 500 tokens in, 100 tokens out
TOKENS_IN_PER_MSG  = 500
TOKENS_OUT_PER_MSG = 100
N_MSGS_PER_MONTH   = 50_000

print(f"Cost per provider for {N_MSGS_PER_MONTH:,} messages/month\n"
      f"  ({TOKENS_IN_PER_MSG} in, {TOKENS_OUT_PER_MSG} out per message):\n")
print(f"  {'Provider':<32}{'monthly $':>12}")
print(f"  {'-'*32}{'-'*12}")
for name, p in PRICES.items():
    total = N_MSGS_PER_MONTH * (TOKENS_IN_PER_MSG/1000 * p["in"]
                                + TOKENS_OUT_PER_MSG/1000 * p["out"])
    print(f"  {name:<32}${total:>10,.2f}")


Cost per provider for 50,000 messages/month
  (500 in, 100 out per message):

  Provider                           monthly $
  --------------------------------------------
  OpenAI gpt-5.4-mini             $     41.25
  OpenAI gpt-5.5                  $    275.00
  Anthropic claude-haiku          $     50.00
  Anthropic claude-sonnet         $    150.00
  Google gemini-2.5-flash         $     20.00
  Google gemini-2.5-pro           $     81.25
  Ollama llama3.2:3b (local)      $      0.00


**The takeaways.** At ~50K messages/month with these per-message sizes:

- **Gemini Flash** is the cheapest hosted option (~$20/mo).
- **gpt-5.4-mini** is competitive on cost and very reliable.
- **Flagship models** (gpt-5.5, claude-sonnet) are 3–7× more expensive — pay for them only when the small models fail your eval.
- **Local (Ollama)** is free per call but ties up your hardware. The break-even depends on your usage; at ~100K calls/month, a $400 GPU pays for itself in under a year.

---

### ✋ Quick exercise (~2 min) — Cost math for two models

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Your team expects **10,000 support messages per month**, each about **800 tokens in / 200 tokens out**. Reusing the `PRICES` dict and the cost formula from §9, compute the monthly cost for `"OpenAI gpt-5.4-mini"` and `"Google gemini-2.5-flash"`. Which is cheaper?

```python
N, T_IN, T_OUT = 10_000, 800, 200
```

In [16]:
# ✍️ Your turn 👇
# Reuse the PRICES dict and cost formula from §9.
# Workload: 10,000 messages/month, 800 tokens in, 200 tokens out each.
# Compare "OpenAI gpt-5.4-mini" vs "Google gemini-2.5-flash" — which is cheaper?
N, T_IN, T_OUT = 10_000, 800, 200


<details>
<summary>✅ <b>Solution</b></summary>

```python
N, T_IN, T_OUT = 10_000, 800, 200
for name in ("OpenAI gpt-5.4-mini", "Google gemini-2.5-flash"):
    p = PRICES[name]
    cost = N * (T_IN/1000 * p["in"] + T_OUT/1000 * p["out"])
    print(f"{name:<28} ${cost:.2f}")
# OpenAI gpt-5.4-mini          $15.00
# Google gemini-2.5-flash      $7.40
```

Reusing the §9 formula `cost = (tokens_in/1000)*price_in + (tokens_out/1000)*price_out`: **gemini-2.5-flash** comes out at about half the cost (~$7.40 vs ~$15.00/mo). As §9 stresses, cheaper only wins if it also passes your eval — but here Flash is both cheaper and a perfectly capable classifier.
</details>

## 10. The embedding side — same idea for NB 23

Embeddings (the vector representations used in retrieval) have the same hosted-vs-local choice.

In [17]:
from llm_providers import MockEmbedder, get_embedder

emb = MockEmbedder(dim=128)
vecs = emb.embed([
    "How do I cancel my subscription?",
    "How can I end my plan?",
    "How is the weather today?",
])
print(f"shape: {vecs.shape}")
# Cosine similarity (L2-normalised → dot product)
import numpy as np
print(f"sim('cancel sub', 'end plan') = {float(vecs[0] @ vecs[1]):.3f}")
print(f"sim('cancel sub', 'weather')  = {float(vecs[0] @ vecs[2]):.3f}")


shape: (3, 128)
sim('cancel sub', 'end plan') = 0.574
sim('cancel sub', 'weather')  = 0.194


### Real embedders

| Embedder | Model | Cost | Where it runs |
|---|---|---|---|
| `OpenAIEmbedder` | `text-embedding-3-small` | $0.00002 / 1K tokens (very cheap) | OpenAI cloud |
| `OpenAIEmbedder` | `text-embedding-3-large` | $0.00013 / 1K tokens | OpenAI cloud |
| `LocalEmbedder` | `all-MiniLM-L6-v2` (80 MB) | free per call | your machine |
| `LocalEmbedder` | `BAAI/bge-large-en-v1.5` (1.3 GB) | free per call | your machine |

```python
from llm_providers import OpenAIEmbedder, LocalEmbedder

# Hosted:
emb = OpenAIEmbedder("text-embedding-3-small")

# Local (one-time download on first call):
# pip install sentence-transformers
emb = LocalEmbedder("all-MiniLM-L6-v2")

vectors = emb.embed(["my first doc", "my second doc"])
```

> 💡 **For most semantic-search tasks, `all-MiniLM-L6-v2` is good enough.** It's 22M parameters, runs on a CPU, and matches the quality of much larger models for short-to-medium texts.

## 11. Swapping into the existing AI notebooks

Every AI notebook (NB 21–26, and the NB 42 capstone) defines an `llm = MockLLM()` near the top — either imported from `llm_providers.py` or vendored inline. Replace exactly that one line:

```python
# Before:
from llm_providers import MockLLM
llm = MockLLM()

# After (any one of these):
from llm_providers import OpenAILLM
llm = OpenAILLM(model="gpt-5.4-mini")

from llm_providers import AnthropicLLM
llm = AnthropicLLM(model="claude-haiku-4-5")

from llm_providers import GoogleLLM
llm = GoogleLLM(model="gemini-2.5-flash")

from llm_providers import OllamaLLM
llm = OllamaLLM(model="llama3.2:3b")
```

The rest of the notebook is unchanged. That's the whole point of the unified interface.

> ⚠️ **Heads-up:** real LLMs are non-deterministic by default. Cell outputs in the existing notebooks (which were captured against `MockLLM`) will *not* match exactly when you swap to a real provider. That is *correct* — and exactly what the eval harness in NB 26 is for.

## 12. Production patterns you'll want eventually

A few patterns that pay back in real deployments:

| Pattern | What it does | Where to add it |
|---|---|---|
| **Caching** by `(prompt_hash, model)` | Skip the API call if you've seen the exact prompt before | A `functools.lru_cache` around `chat()` |
| **Retry with exponential backoff** | Survive transient 429 / 5xx errors | NB 12 has the reusable function |
| **Cost ceiling per request** | Refuse calls that would exceed a hard cap | Wrap the constructor in a factory |
| **Provider fallback chain** | Try Anthropic; on failure, fall back to OpenAI | A small loop over `[primary, fallback]` |
| **Streaming responses** | Show tokens as they generate (UX!) | Provider-specific; OpenAI / Anthropic both support it |

These are out of scope for the course's notebooks (which prioritise the *patterns*), but every one of them is 10–30 lines of code on top of what you have.

## 🧪 Exercises

### Exercise 1 — A provider-swap drill

Open NB 22 (`06_ai_engineering/22_ai_workflows.ipynb`). Find the cell that creates `llm = MockLLM()`. Change it to use `OpenAILLM(model="gpt-5.4-mini")` *as a code comment only* — do not actually run it unless you have a key. Then write a 3-line cell beneath it that prints which provider is active.

In [18]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
# Change the original line to:
# from llm_providers import OpenAILLM
# llm = OpenAILLM(model="gpt-5.4-mini")

# Then add this confirmation cell:
print(f"Provider: {type(llm).__name__}")
print(f"Model   : {getattr(llm, 'model', 'mock-mini')}")
```

You'd see `Provider: OpenAILLM, Model: gpt-5.4-mini` after the swap. Every other cell in the notebook continues to work without change — that's the value of the unified interface.
</details>

### Exercise 2 — Estimate cost for *your* expected workload

Pick a workload: say, **20,000 customer-feedback messages per month**, average **300 tokens in / 80 tokens out**. Estimate the monthly cost for each of the four hosted models in §9. Which is cheapest? Which would you actually ship and why?

In [19]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
TOKENS_IN, TOKENS_OUT, N = 300, 80, 20_000
for name, p in PRICES.items():
    cost = N * (TOKENS_IN/1000 * p["in"] + TOKENS_OUT/1000 * p["out"])
    print(f"{name:<32}${cost:.2f}")
```

You'll find Gemini Flash and `gpt-5.4-mini` come out closest. Which one to ship depends on:

- **Quality on your eval set** (NB 26). The cheapest model that passes is the winner.
- **Vendor risk.** OpenAI has the longest track record but Google's SLA is excellent.
- **Existing infrastructure.** If your team is already on GCP, Gemini is a one-click integration.

This is exactly the kind of analysis to write up before any AI feature ships.
</details>

### Exercise 3 — A local-first development workflow

Imagine you're prototyping an AI feature on a flight (no internet). Write a function `local_first_chat(messages, prefer_local: bool = True)` that:

1. If `prefer_local` is True and `OllamaLLM` is available, use it.
2. Otherwise fall back to `OpenAILLM` if `OPENAI_API_KEY` is set.
3. Otherwise fall back to `MockLLM`.

Always return the same response dict.

In [20]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
import os
from llm_providers import MockLLM, OpenAILLM, OllamaLLM

def local_first_chat(messages, *, prefer_local: bool = True, **kwargs):
    if prefer_local:
        try:
            return OllamaLLM(model="llama3.2:3b").chat(messages, **kwargs)
        except Exception:
            pass    # Ollama not installed / not running → fall through
    if os.getenv("OPENAI_API_KEY"):
        try:
            return OpenAILLM(model="gpt-5.4-mini").chat(messages, **kwargs)
        except Exception:
            pass
    return MockLLM().chat(messages, **kwargs)


# Should pick the lowest-friction available option without crashing.
r = local_first_chat([{"role": "user", "content": "Hi!"}])
print(f"Used model: {r['model']}")
```

This is the pattern behind every "works offline, syncs when online" tool. The function tries options in order of *preference*; each attempt fails fast on a clean `ImportError` or `RuntimeError`, never throwing a traceback at the user.
</details>

## 🧠 Key takeaways

1. The course uses **`MockLLM` by default** so every notebook runs offline.
2. A single module (`llm_providers.py`) gives all five providers the **same `chat()` interface**.
3. **Swapping providers** is a **one-line change** in any of the AI notebooks.
4. **OpenAI** is the safest default; **Anthropic** for long context / careful drafts; **Gemini** for cost-sensitive high-volume; **Ollama** for offline / privacy.
5. **Always validate on your golden set** (NB 26) when comparing providers — cost without accuracy is meaningless.
6. **The smallest model that passes the eval wins.** Don't over-pay for capacity you don't use.
7. **Embeddings** have the same hosted-vs-local choice — pick `OpenAIEmbedder` or `LocalEmbedder` (sentence-transformers).
8. **Never commit API keys.** Use environment variables (`os.getenv`) or `.env` files.

## ✅ Self-assessment

- [ ] Pick the right provider for a given cost / privacy / latency requirement
- [ ] Install and authenticate any of OpenAI / Anthropic / Gemini
- [ ] Install Ollama and pull a local model
- [ ] Swap the LLM provider in any AI notebook with one line
- [ ] Estimate monthly cost for a given workload across providers
- [ ] Build a fallback chain (`local → hosted → mock`)

## 🚀 Where to go from here

You now have a portable LLM toolkit. The natural next steps:

1. **Set up a real key** for one hosted provider and re-run NB 26's evaluation harness against it. Compare with the MockLLM numbers.
2. **Try a local model** via Ollama on NB 22's inbox-triage task. Note the latency difference vs. the mock.
3. **Build a fallback** — your production code should try a fast provider, fall back to a robust one, never to a crash.
